In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.optimizers import Adam
import json
import os
import pickle
import optuna
from sklearn.utils.class_weight import compute_class_weight

ModuleNotFoundError: No module named 'optuna'

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

In [ ]:
RAND = 1337
np.random.seed(RAND)

In [ ]:
!curl -L -o dataset.zip https://www.kaggle.com/api/v1/datasets/download/chethuhn/network-intrusion-dataset
!unzip dataset.zip
!rm dataset.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  229M  100  229M    0     0  82.8M      0  0:00:02  0:00:02 --:--:-- 97.9M
Archive:  dataset.zip
  inflating: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  
  inflating: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  
  inflating: Friday-WorkingHours-Morning.pcap_ISCX.csv  
  inflating: Monday-WorkingHours.pcap_ISCX.csv  
  inflating: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  
  inflating: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  
  inflating: Tuesday-WorkingHours.pcap_ISCX.csv  
  inflating: Wednesday-workingHours.pcap_ISCX.csv  


In [ ]:
!ls -lh

total 844M
-rw-r--r-- 1 root root  74M Aug 28  2023 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
-rw-r--r-- 1 root root  74M Aug 28  2023 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
-rw-r--r-- 1 root root  56M Aug 28  2023 Friday-WorkingHours-Morning.pcap_ISCX.csv
-rw-r--r-- 1 root root 169M Aug 28  2023 Monday-WorkingHours.pcap_ISCX.csv
drwxr-xr-x 1 root root 4.0K May 26 13:25 sample_data
-rw-r--r-- 1 root root  80M Aug 28  2023 Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
-rw-r--r-- 1 root root  50M Aug 28  2023 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
-rw-r--r-- 1 root root 129M Aug 28  2023 Tuesday-WorkingHours.pcap_ISCX.csv
-rw-r--r-- 1 root root 215M Aug 28  2023 Wednesday-workingHours.pcap_ISCX.csv


In [ ]:
train_files = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'
]

# Test: Sexta completa
test_files = [
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',          # Botnet
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', # PortScan
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'      # DDoS
]

df_train_full = pd.concat([pd.read_csv(f) for f in train_files])
df_test = pd.concat([pd.read_csv(f) for f in test_files])


NameError: name 'RAND' is not defined

In [ ]:
df_train_full.columns = df_train_full.columns.str.strip()
df_test.columns = df_test.columns.str.strip()

In [ ]:
df_train_full.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [ ]:
df_test.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,3268,112740690,32,16,6448,1152,403,0,201.5,204.724205,...,32,3.594286e+02,1.199802e+01,380,343,16100000.0,4.988048e+05,16400000,15400000,BENIGN
1,389,112740560,32,16,6448,5056,403,0,201.5,204.724205,...,32,3.202857e+02,1.574499e+01,330,285,16100000.0,4.987937e+05,16400000,15400000,BENIGN
2,0,113757377,545,0,0,0,0,0,0.0,0.000000,...,0,9.361829e+06,7.324646e+06,18900000,19,12200000.0,6.935824e+06,20800000,5504997,BENIGN
3,5355,100126,22,0,616,0,28,28,28.0,0.000000,...,32,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN
4,0,54760,4,0,0,0,0,0,0.0,0.000000,...,0,0.000000e+00,0.000000e+00,0,0,0.0,0.000000e+00,0,0,BENIGN


In [ ]:
df_train_full.describe()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,...,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06,2.127498e+06
mean,8.107429e+03,1.617974e+07,1.022903e+01,1.139636e+01,5.458908e+02,1.795941e+04,1.924798e+02,1.840121e+01,5.204267e+01,6.106676e+01,...,5.698084e+00,-3.656445e+03,7.658728e+04,4.543493e+04,1.547773e+05,5.077570e+04,9.402080e+06,2.631721e+05,9.618372e+06,9.171892e+06
std,1.860855e+04,3.517357e+07,7.996815e+02,1.061514e+03,1.120836e+04,2.414015e+06,5.093888e+02,4.190767e+01,1.247679e+02,1.759272e+02,...,6.637941e+02,1.251528e+06,6.275481e+05,4.122672e+05,1.039559e+06,5.475768e+05,2.560104e+07,3.030327e+06,2.594098e+07,2.547929e+07
min,0.000000e+00,-4.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,1.710000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,3.151000e+04,2.000000e+00,2.000000e+00,6.800000e+01,1.420000e+02,4.100000e+01,2.000000e+00,3.700000e+01,0.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,4.220048e+06,5.000000e+00,4.000000e+00,3.340000e+02,5.570000e+02,2.090000e+02,3.800000e+01,5.200000e+01,7.673470e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.293000e+03,4.672000e+03,7.125597e+03,...,2.135570e+05,1.380000e+02,1.070000e+08,7.420000e+07,1.070000e+08,1.070000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [ ]:
df_test.describe()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,703245.000000,7.032450e+05,703245.000000,703245.000000,7.032450e+05,7.032450e+05,703245.000000,703245.000000,703245.000000,703245.000000,...,703245.000000,703245.000000,7.032450e+05,7.032450e+05,7.032450e+05,7.032450e+05,7.032450e+05,7.032450e+05,7.032450e+05,7.032450e+05
mean,7962.735034,1.056822e+07,6.735627,7.360662,5.596234e+02,1.072696e+04,253.342164,19.658897,76.835304,92.638353,...,4.571550,25.690055,9.656884e+04,2.812304e+04,1.483580e+05,8.104613e+04,5.030473e+06,1.231939e+06,5.904587e+06,4.132828e+06
std,17263.042486,2.815051e+07,572.363391,771.589746,4.685555e+03,1.728003e+06,1132.537365,96.648907,303.048257,473.159672,...,545.328536,7.290533,7.082775e+05,3.293842e+05,9.830942e+05,6.578535e+05,1.582620e+07,7.536521e+06,1.854219e+07,1.463286e+07
min,0.000000,-1.300000e+01,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,53.000000,6.800000e+01,1.000000,1.000000,2.000000e+00,6.000000e+00,2.000000,0.000000,2.000000,0.000000,...,0.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,80.000000,3.074500e+04,2.000000,1.000000,3.000000e+01,5.600000e+01,20.000000,2.000000,8.666667,0.000000,...,1.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,3324.000000,1.810259e+06,4.000000,4.000000,8.000000e+01,3.480000e+02,45.000000,31.000000,42.000000,10.263203,...,3.000000,32.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,65532.000000,1.200000e+08,207964.000000,284602.000000,1.235152e+06,6.270000e+08,24820.000000,2325.000000,5940.857143,7049.469004,...,198636.000000,60.000000,1.100000e+08,7.050000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.660000e+07,1.200000e+08,1.200000e+08


In [ ]:
df_train_full.isna().sum().sum(), df_test.isna().sum().sum()

(np.int64(1311), np.int64(47))

In [ ]:
df_train_full['Flow Bytes/s'].isna().sum(), df_train_full['Flow Bytes/s'].shape

(np.int64(1311), (2127498,))

In [ ]:
df_test['Flow Bytes/s'].isna().sum(), df_test['Flow Bytes/s'].shape

(np.int64(47), (703245,))

In [ ]:
df_train_full[df_train_full['Flow Bytes/s'].isna()]['Label'].value_counts()

,count
Label,
DoS Hulk,949
BENIGN,362


In [ ]:
df_test[df_test['Flow Bytes/s'].isna()]['Label'].value_counts()

,count
Label,
BENIGN,47


In [ ]:
df_train_full[df_train_full['Label'] == 'DoS Hulk'].shape

(231073, 79)

In [ ]:
df_test[df_test['Label'] == 'BENIGN'].shape

(414322, 79)

In [ ]:
df_train_full = df_train_full.dropna(subset=['Flow Bytes/s'])
df_test = df_test.dropna(subset=['Flow Bytes/s'])

In [ ]:
df_train_full.isna().sum().sum(), df_test.isna().sum().sum()

(np.int64(0), np.int64(0))

In [ ]:
logging.info("Data Preprocessing...")

In [ ]:
df_train_full['Label'] = df_train_full['Label'].apply(lambda x: 0 if x == "BENIGN" else 1)
df_test['Label'] = df_test['Label'].apply(lambda x: 0 if x == "BENIGN" else 1)
# Class 0 -> Benign
# Class 1 -> Attack

In [ ]:
df_train_full['Label'].unique(), df_test['Label'].unique()

(array([0, 1]), array([0, 1]))

In [ ]:
np.isinf(df_train_full).sum().sum(), np.isinf(df_test).sum().sum()

(np.int64(2058), np.int64(960))

In [ ]:
df_train_full = df_train_full.replace([np.inf, -np.inf], np.nan)
df_test = df_test.replace([np.inf, -np.inf], np.nan)
df_train_full = df_train_full.dropna(subset=['Flow Bytes/s'])
df_test = df_test.dropna(subset=['Flow Bytes/s'])

In [ ]:
# Val: 20% do treino
df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=RAND)

In [ ]:
X_train = df_train.drop('Label', axis=1)
y_train = df_train['Label']
X_val = df_val.drop('Label', axis=1)
y_val = df_val['Label']
X_test = df_test.drop('Label', axis=1)
y_test = df_test['Label']

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

logging.info(f"Class weights: {class_weight_dict}")


In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((1700126, 78), (1700126,), (425032, 78), (425032,), (702718, 78), (702718,))

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1700126 entries, 198794 to 339855
Data columns (total 78 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  F

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),
         make_column_selector(dtype_include=['number']))
    ])

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

In [ ]:
with open('./models/preprocessor_v1.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

In [ ]:

logging.info(f"X_train shape: {X_train.shape}")

In [ ]:
BATCH_SIZE=1024
EPOCHS=1000

In [ ]:
strategy = tf.distribute.MirroredStrategy()

def objective(trial):
  with strategy.scope():
    model = Sequential()
    model.add(Dense(1024,input_dim=X_train.shape[1],activation='relu'))
    model.add(Dropout(0.01))
    model.add(Dense(768,activation='relu'))
    model.add(Dropout(0.01))
    model.add(Dense(512,activation='relu'))
    model.add(Dropout(0.01))
    model.add(Dense(1, activation='sigmoid'))

    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=lr),metrics=['accuracy'])

  early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,
      restore_best_weights=True
  )


  model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=0
  )


  loss, accuracy = model.evaluate(X_val, y_val, verbose=0)

  return accuracy

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

In [ ]:
logging.info("Best hyperparameters:")
for key, value in study.best_params.items():
  logging.info(f"{key}: {value}")

In [ ]:
best_lr = study.best_params['lr']

In [ ]:
checkpoint = ModelCheckpoint(
    './checkpoints/checkpoint.keras',
    monitor='val_loss',
    save_best_only=True
)


In [ ]:
logging.info("Starting model training")

In [ ]:
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
  model = Sequential()
  model.add(Dense(1024,input_dim=X_train.shape[1],activation='relu'))
  model.add(Dropout(0.01))
  model.add(Dense(768,activation='relu'))
  model.add(Dropout(0.01))
  model.add(Dense(512,activation='relu'))
  model.add(Dropout(0.01))
  model.add(Dense(1, activation='sigmoid'))

  model.summary()

  model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=best_lr),metrics=['accuracy'])

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 1024)           │        80,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 768)            │       787,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           513 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,262,337 (4.82 MB)

 Trainable params: 1,262,337 (4.82 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
  X_train, y_train,
  batch_size=BATCH_SIZE,
  epochs=EPOCHS,
  validation_data=(X_val, y_val),
  callbacks=[early_stop, checkpoint],
  verbose=1
)
model.save('./models/model_cic-ids-2017-v2.keras')

31814/31814 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9714 - loss: 0.0671

31814/31814 ━━━━━━━━━━━━━━━━━━━━ 557s 18ms/step - accuracy: 0.9761 - loss: 0.0553 - val_accuracy: 0.9779 - val_loss: 0.0476


In [ ]:
logging.info("Done. Evaluating")

In [ ]:

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred , average="binary")
precision = precision_score(y_test, y_pred , average="binary")
f1 = f1_score(y_test, y_pred, average="binary")
print("----------------------------------------------")
print("accuracy")
print("%.3f" %accuracy)
print("recall")
print("%.3f" %recall)
print("precision")
print("%.3f" %precision)
print("f1score")
print("%.3f" %f1)

In [ ]:
results = {
    "accuracy": float(accuracy),
    "recall": float(recall),
    "precision": float(precision),
    "f1": float(f1)
}
with open('./models/results_model_cic-ids-2017-v2.json', 'w') as f:
    json.dump(results, f, indent=2)